In [3]:
!python --version

Python 3.13.12


In [4]:
import torch 

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
True
NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
import os
from ultralytics import YOLO

model = YOLO(r'model\license-plate-finetune-v1n.pt')
model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_

In [5]:
results = model.predict(
    source=r'license_plate.jpg',
    save=True,
    project=r'C:\Auto\license_plate_OCR\runs',
    name='detect',
    exist_ok=True,
    device='cuda'
)


image 1/1 c:\Auto\license_plate_OCR\license_plate.jpg: 288x640 1 License_Plate, 86.1ms
Speed: 3.5ms preprocess, 86.1ms inference, 10.7ms postprocess per image at shape (1, 3, 288, 640)
Results saved to C:\Auto\license_plate_OCR\runs\detect


In [6]:
results[0].speed # milliseconds per image

{'preprocess': 3.532999995513819,
 'inference': 86.07019999908516,
 'postprocess': 10.676899997633882}

In [7]:
import numpy as np

boxes = results[0].boxes.xyxy.cpu().numpy()
scores = results[0].boxes.conf.cpu().numpy()
classes = results[0].boxes.cls.cpu().numpy()

for i in range(len(boxes)):
    x1, y1, x2, y2 = boxes[i]
    conf = scores[i]
    cls = int(classes[i])
    label = results[0].names[cls]

    print({
        "bbox": [float(x1), float(y1), float(x2), float(y2)],
        "confidence": float(conf),
        "class": label
    })

{'bbox': [20.065673828125, 11.4803466796875, 1276.019775390625, 537.2929077148438], 'confidence': 0.8867907524108887, 'class': 'License_Plate'}


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="API_KEY")
project = rf.workspace("traffic-jbhvf").project("thai-license-plate-wniws-h1drt")
version = project.version(2)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to thai-license-plate-2 in yolov11:: 100%|██████████| 311/311 [00:00<00:00, 1906.79it/s]


In [8]:
models = os.listdir("model")
models

['license-plate-finetune-v1l.pt',
 'license-plate-finetune-v1m.pt',
 'license-plate-finetune-v1n.pt',
 'license-plate-finetune-v1s.pt',
 'license-plate-finetune-v1x.pt']

In [9]:
import pandas as pd

results_list = []

for model_file in models:
    model_path = os.path.join("model", model_file)
    model = YOLO(model_path)

    metrics = model.val(
        data=r'thai-license-plate-2\data.yaml',
        split='val',
        imgsz=640,
        device='cuda',
        project=r'C:\Auto\license_plate_OCR\runs',
        name=model_file.split('.')[0].split('-')[-1],
    )

    results_list.append({
        "model": model_file,
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
        "precision": metrics.box.mp,
        "recall": metrics.box.mr,
        "preprocess_ms": metrics.speed['preprocess'],
        "inference_ms": metrics.speed['inference'],
        "postprocess_ms": metrics.speed['postprocess']
    })

df = pd.DataFrame(results_list)

Ultralytics 8.4.32  Python-3.13.12 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLO11l summary (fused): 190 layers, 25,280,083 parameters, 0 gradients, 86.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 202.635.7 MB/s, size: 49.4 KB)
val: Scanning C:\Auto\license_plate_OCR\thai-license-plate-2\valid\labels.cache... 150 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150 30.0Mit/s 0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 135, len(boxes) = 149. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.7it/s 5.7s.6ss
                   all        150        149      0.813      0.826      0.777      0.198
Speed: 2.8ms preprocess, 29.3ms inference, 0.0ms loss, 1.4ms postprocess per i

In [10]:
df

,model,mAP50,mAP50-95,precision,recall,preprocess_ms,inference_ms,postprocess_ms
0,license-plate-finetune-v1l.pt,0.776995,0.198346,0.812648,0.825503,2.788645,29.337597,1.360229
1,license-plate-finetune-v1m.pt,0.837373,0.250691,0.859118,0.852349,1.902029,21.427425,1.028365
2,license-plate-finetune-v1n.pt,0.821776,0.222137,0.886549,0.891582,1.579925,5.745267,1.727384
3,license-plate-finetune-v1s.pt,0.712107,0.214233,0.777976,0.823103,1.689820,12.050245,0.770911
4,license-plate-finetune-v1x.pt,0.880419,0.241383,0.894333,0.919463,1.336167,49.824364,1.048181


In [11]:
df["model_short"] = df["model"].str.extract(r'(v1\w)')

order = ["v1n", "v1s", "v1m", "v1l", "v1x"]
df["model_short"] = pd.Categorical(df["model_short"], categories=order, ordered=True)

In [13]:
df.drop(columns=["model"], inplace=True)
df

,mAP50,mAP50-95,precision,recall,preprocess_ms,inference_ms,postprocess_ms,model_short
0,0.776995,0.198346,0.812648,0.825503,2.788645,29.337597,1.360229,v1l
1,0.837373,0.250691,0.859118,0.852349,1.902029,21.427425,1.028365,v1m
2,0.821776,0.222137,0.886549,0.891582,1.579925,5.745267,1.727384,v1n
3,0.712107,0.214233,0.777976,0.823103,1.689820,12.050245,0.770911,v1s
4,0.880419,0.241383,0.894333,0.919463,1.336167,49.824364,1.048181,v1x


In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

palette = {
    "mAP50":     "#3B8BD4",
    "mAP50-95":  "#5DCAA5",
    "Precision": "#EF9F27",
    "Recall":    "#E24B4A",
}

# melt wide df into long form for seaborn
df_acc = df.melt(
    id_vars="model_short",
    value_vars=["mAP50", "mAP50-95", "precision", "recall"],
    var_name="Metric",
    value_name="Score",
)
df_acc["Metric"] = df_acc["Metric"].str.replace("precision", "Precision").str.replace("recall", "Recall")

sns.set_theme(style="whitegrid", font_scale=1.05)
fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(data=df_acc, x="model_short", y="Score", hue="Metric",
            palette=palette, ax=ax, edgecolor="none", width=0.75)

ax.set_title("BBox Accuracy by Model  |  150 images · CUDA",
             fontweight="bold", pad=12)
ax.set_xlabel("")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.legend(title="", fontsize=10, loc="upper right")

# fix: get x position from the bar's actual position in the grouped plot
best_idx = df["mAP50"].idxmax()
best_model = df.loc[best_idx, "model_short"]
models_order = df["model_short"].tolist()
x_pos = models_order.index(best_model)
ax.annotate("★ best mAP50",
            xy=(x_pos, df.loc[best_idx, "mAP50"] + 0.02),
            fontsize=8.5, color="#3B8BD4", ha="center")

plt.tight_layout()
plt.savefig("./image/chart1_accuracy.png", dpi=150, bbox_inches="tight")

C:\Users\Auto\AppData\Local\Temp\ipykernel_19420\4258237722.py:44: UserWarning: Glyph 9733 (\N{BLACK STAR}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\Auto\AppData\Local\Temp\ipykernel_19420\4258237722.py:45: UserWarning: Glyph 9733 (\N{BLACK STAR}) missing from font(s) Arial.
  plt.savefig("./image/chart1_accuracy.png", dpi=150, bbox_inches="tight")


In [15]:
import numpy as np

size_order = ["v1n", "v1s", "v1m", "v1l", "v1x"]

# fix: sort by model_short, not model
df_speed = df.set_index("model_short").loc[size_order].reset_index()

speed_cols   = ["inference_ms",  "preprocess_ms",  "postprocess_ms"]
speed_colors = ["#7F77DD",       "#D4537E",         "#888780"]
speed_labels = ["Inference",     "Preprocess",      "Postprocess"]

sns.set_theme(style="whitegrid", font_scale=1.05)
fig, ax = plt.subplots(figsize=(9, 5))

bottoms = np.zeros(len(df_speed))
for col, color, label in zip(speed_cols, speed_colors, speed_labels):
    ax.bar(df_speed["model_short"], df_speed[col], bottom=bottoms,
           color=color, label=label, edgecolor="none", width=0.6)
    bottoms += df_speed[col].values

for i, (_, row) in enumerate(df_speed.iterrows()):
    total = row["inference_ms"] + row["preprocess_ms"] + row["postprocess_ms"]
    ax.text(i, total + 0.4, f"{total:.1f} ms",
            ha="center", fontsize=8.5, color="#444")

ax.set_title("Inference Speed by Model  |  150 images · CUDA  (lower = faster)",
             fontweight="bold", pad=12)
ax.set_xlabel("")
ax.set_ylabel("Milliseconds (ms)")
ax.legend(title="", fontsize=10, loc="upper left")
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.savefig("./image/chart2_speed.png", dpi=150, bbox_inches="tight")
plt.show()

<Figure size 900x500 with 1 Axes>

<Figure size 900x500 with 1 Axes>

In [16]:
color_list = ["#3B8BD4", "#5DCAA5", "#EF9F27", "#E24B4A", "#7F77DD"]

label_offsets = {
    "v1l": ( 0.8,  0.003),
    "v1m": ( 0.8,  0.003),
    "v1n": ( 0.8, -0.011),
    "v1s": ( 0.8,  0.003),
    "v1x": ( 0.8,  0.003),
}

sns.set_theme(style="whitegrid", font_scale=1.05)
fig, ax = plt.subplots(figsize=(9, 6))

for i, row in df.iterrows():
    ax.scatter(row["inference_ms"], row["mAP50"],
               s=200, color=color_list[i], zorder=3,
               edgecolors="white", linewidths=1.0, label=row["model_short"])
    dx, dy = label_offsets[row["model_short"]]
    ax.annotate(row["model_short"],
                xy=(row["inference_ms"], row["mAP50"]),
                xytext=(row["inference_ms"] + dx, row["mAP50"] + dy),
                fontsize=10, color=color_list[i], fontweight="bold")

ax.axvline(df["inference_ms"].mean(), color="gray",
           linestyle="--", linewidth=0.8, alpha=0.5, label="avg speed")
ax.axhline(df["mAP50"].mean(), color="gray",
           linestyle=":",  linewidth=0.8, alpha=0.5, label="avg mAP50")

ax.set_title("Accuracy vs Speed Trade-off  |  150 images · CUDA",
             fontweight="bold", pad=12)
ax.set_xlabel("Inference time (ms)  →  lower is faster")
ax.set_ylabel("mAP50  →  higher is better")
ax.set_xlim(-2, 62)
# fix: use actual data range with padding instead of wrong (0.35, 0.54)
ax.set_ylim(df["mAP50"].min() - 0.05, df["mAP50"].max() + 0.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.legend(fontsize=9, loc="lower right")

plt.tight_layout()
plt.savefig("./image/chart3_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

<Figure size 900x600 with 1 Axes>

In [17]:
model = YOLO(r'model\license-plate-finetune-v1n.pt')
model.predict(
    source=r'thai-license-plate-2\valid\images\4f94769eb2f06e2f1753f0a5bd73bef6_jpg.rf.08a3cfcd26dd4a04f777dbe351bb7807.jpg',
    save=True,
    project=r'C:\Auto\license_plate_OCR\runs',
    name='detect',
    exist_ok=True,
    device='cuda',
    verbose=False
)

Results saved to C:\Auto\license_plate_OCR\runs\detect


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'License_Plate'}
 obb: None
 orig_img: array([[[124, 138, 150],
         [122, 137, 146],
         [116, 133, 142],
         ...,
         [221, 190, 159],
         [221, 190, 159],
         [221, 190, 159]],
 
        [[144, 156, 168],
         [139, 151, 161],
         [ 90, 105, 114],
         ...,
         [220, 189, 158],
         [220, 189, 158],
         [220, 189, 158]],
 
        [[135, 143, 156],
         [130, 139, 149],
         [ 87,  96, 106],
         ...,
         [220, 189, 158],
         [220, 189, 158],
         [220, 189, 158]],
 
        ...,
 
        [[208, 231, 246],
         [187, 210, 225],
         [194, 217, 232],
         ...,
         [237, 250, 252],
         [223, 236, 238],
         [209, 222, 224]],
 
        [[193, 214, 229],
         [182, 203, 218],
         [200, 221, 236],
         ...,
         [2

In [5]:
from ultralytics import YOLO

model = YOLO(r'model\license-plate-finetune-v1n.pt')

metrics = model.val(
    data=r'License-Plate-Recognition-11\data.yaml',
    split='test',
    imgsz=320,
    device='cuda',
    project=r'C:\Auto\license_plate_OCR\runs',
    name="nano_test",
)

last = {
    "model": "license-plate-finetune-v1n.pt",
    "mAP50": metrics.box.map50,
    "mAP50-95": metrics.box.map,
    "precision": metrics.box.mp,
    "recall": metrics.box.mr,
    "preprocess_ms": metrics.speed['preprocess'],
    "inference_ms": metrics.speed['inference'],
    "postprocess_ms": metrics.speed['postprocess']
}

last

AttributeError: module 'torch' has no attribute '_utils'